# Projet de Stage Technique - EST (DUT)
## Nettoyage et Préparation du Jeu de Données de Maintenance des Camions

---

**Objectif :**  
Ce notebook présente les étapes de nettoyage, de standardisation et d'enrichissement
du jeu de données de maintenance de la flotte de camions Volvo FH.
Le résultat final est un fichier CSV exploitable par un système de maintenance prédictive basé sur l'IA.

**Plan du notebook :**
1. Importation des bibliothèques et chargement des données
2. Renommage des colonnes
3. Exploration et analyse initiale
4. Nettoyage des données (doublons, valeurs nulles, aberrantes)
5. Filtrage et sélection du sous-ensemble d'étude
6. Standardisation des variables catégorielles
7. Génération des variables manquantes (simulation réaliste)
8. Export CSV et génération du rapport PDF

---

In [1]:
# =============================================================
# Projet de Stage Technique - EST (DUT)
# Module : Traitement et Nettoyage des Données
# Fichier : Nettoyage du jeu de données de maintenance des camions
# =============================================================

# --- Importation des bibliothèques nécessaires ---
import pandas as pd
import random
import warnings
warnings.filterwarnings('ignore')

print('Bibliothèques importées avec succès.')

# --- Chargement du jeu de données brut ---
CSV_INPUT = 'logistics_dataset_with_maintenance_required.csv'
df = pd.read_csv(CSV_INPUT, delimiter=';')
print(f'Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes.')


Bibliothèques importées avec succès.
Données chargées : 92000 lignes, 27 colonnes.


In [2]:
# --- Renommage des colonnes (anglais -> français) ---
# Objectif : uniformiser les noms de colonnes en français
# pour faciliter la lecture et l'interprétation des données.

new_columns = {
    'Vehicle_ID'          : 'Identifiant_Véhicule',
    'Make_and_Model'      : 'Marque_et_Modèle',
    'Year_of_Manufacture' : 'Année_Fabrication',
    'Vehicle_Type'        : 'Type_Véhicule',
    'Usage_Hours'         : 'Heures_Utilisation',
    'Route_Info'          : 'Info_Itinéraire',
    'Load_Capacity'       : 'Capacité_Charge',
    'Actual_Load'         : 'Charge_Réelle',
    'Last_Maintenance_Date': 'Date_Dernier_Entretien',
    'Maintenance_Type'    : 'Type_Entretien',
    'Maintenance_Cost'    : 'Coût_Entretien',
    'Engine_Temperature'  : 'Température_Moteur',
    'Tire_Pressure'       : 'Pression_Pneus',
    'Fuel_Consumption'    : 'Consommation_Carburant',
    'Battery_Status'      : 'État_Batterie',
    'Vibration_Levels'    : 'Niveaux_Vibration',
    'Oil_Quality'         : 'Qualité_Huile',
    'Brake_Condition'     : 'État_Freins',
    'Failure_History'     : 'Historique_Pannes',
    'Anomalies_Detected'  : 'Anomalies_Détectées',
    'Predictive_Score'    : 'Score_Prédictif',
    'Maintenance_Required': 'Entretien_Nécessaire',
    'Weather_Conditions'  : 'Conditions_Météo',
    'Road_Conditions'     : 'Conditions_Route',
    'Delivery_Times'      : 'Délais_Livraison',
    'Downtime_Maintenance': 'Temps_Arrêt_Entretien',
    'Impact_on_Efficiency': 'Impact_Efficacité',
}

df.rename(columns=new_columns, inplace=True)
print('Colonnes renommées avec succès.')


Colonnes renommées avec succès.


## 2. Exploration Initiale des Données

Avant tout traitement, il est indispensable d'examiner la structure du DataFrame :
types de données, présence de valeurs nulles, et statistiques descriptives.

In [3]:
# --- Aperçu des 5 premières lignes du DataFrame ---
df.head()


,Identifiant_Véhicule,Marque_et_Modèle,Année_Fabrication,Type_Véhicule,Heures_Utilisation,Info_Itinéraire,Capacité_Charge,Charge_Réelle,Date_Dernier_Entretien,Type_Entretien,...,État_Freins,Historique_Pannes,Anomalies_Détectées,Score_Prédictif,Entretien_Nécessaire,Conditions_Météo,Conditions_Route,Délais_Livraison,Temps_Arrêt_Entretien,Impact_Efficacité
0,1,Ford F-150,2022,Truck,530,Rural,7.534549,9.004247,09/04/2023,Oil Change,...,Good,1,0,0.171873,1,Clear,Highway,30.000000,0.093585,0.150063
1,2,Volvo FH,2015,Van,10679,Rural,7.671728,6.111785,20/07/2023,Tire Rotation,...,Fair,1,0,0.246670,1,Clear,Rural,30.000000,3.361201,0.343017
2,3,Chevy Silverado,2022,Van,4181,Rural,2.901159,3.006055,17/03/2023,Oil Change,...,Good,1,1,0.455236,1,Clear,Highway,48.627823,1.365300,0.100000
3,4,Chevy Silverado,2011,Truck,2974,Urban,15.893347,18.825290,01/05/2024,Tire Rotation,...,Good,0,1,0.060208,1,Clear,Highway,30.000000,0.000000,0.135749
4,5,Ford F-150,2014,Van,2539,Rural,60.668320,65.605463,15/11/2023,Tire Rotation,...,Good,1,1,0.264929,1,Rainy,Urban,300.000000,6.608704,0.395193


In [4]:
# --- Informations générales sur le DataFrame ---
print('=== Informations sur les colonnes et les types de données ===')
df.info()

# --- Statistiques descriptives globales ---
print('\n=== Statistiques descriptives ===')
df.describe(include='all')


=== Informations sur les colonnes et les types de données ===
<class 'pandas.DataFrame'>
RangeIndex: 92000 entries, 0 to 91999
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Identifiant_Véhicule    92000 non-null  int64  
 1   Marque_et_Modèle        92000 non-null  str    
 2   Année_Fabrication       92000 non-null  int64  
 3   Type_Véhicule           92000 non-null  str    
 4   Heures_Utilisation      92000 non-null  int64  
 5   Info_Itinéraire         92000 non-null  str    
 6   Capacité_Charge         92000 non-null  float64
 7   Charge_Réelle           92000 non-null  float64
 8   Date_Dernier_Entretien  92000 non-null  str    
 9   Type_Entretien          92000 non-null  str    
 10  Coût_Entretien          92000 non-null  float64
 11  Température_Moteur      92000 non-null  float64
 12  Pression_Pneus          92000 non-null  float64
 13  Consommation_Carburant  92000 non-null  

,Identifiant_Véhicule,Marque_et_Modèle,Année_Fabrication,Type_Véhicule,Heures_Utilisation,Info_Itinéraire,Capacité_Charge,Charge_Réelle,Date_Dernier_Entretien,Type_Entretien,...,État_Freins,Historique_Pannes,Anomalies_Détectées,Score_Prédictif,Entretien_Nécessaire,Conditions_Météo,Conditions_Route,Délais_Livraison,Temps_Arrêt_Entretien,Impact_Efficacité
count,92000.00000,92000,92000.000000,92000,92000.000000,92000,92000.000000,92000.000000,92000,92000,...,92000,92000.000000,92000.000000,92000.000000,92000.000000,92000,92000,92000.000000,92000.000000,92000.000000
unique,NaN,4,NaN,2,NaN,3,NaN,NaN,547,3,...,3,NaN,NaN,NaN,NaN,4,3,NaN,NaN,NaN
top,NaN,Ford F-150,NaN,Van,NaN,Highway,NaN,NaN,26/03/2023,Oil Change,...,Good,NaN,NaN,NaN,NaN,Clear,Highway,NaN,NaN,NaN
freq,NaN,45917,NaN,50850,NaN,46052,NaN,NaN,219,41488,...,45938,NaN,NaN,NaN,NaN,73482,36861,NaN,NaN,NaN
mean,46000.50000,NaN,2016.968478,NaN,2989.550913,NaN,25.068001,23.771355,NaN,NaN,...,NaN,0.399565,0.452196,0.166754,0.767989,NaN,NaN,99.283161,3.210282,0.208325
std,26558.25672,NaN,5.359597,NaN,2992.083426,NaN,25.040153,24.194555,NaN,NaN,...,NaN,0.489812,0.497712,0.103435,0.422118,NaN,NaN,79.708201,6.429751,0.111234
min,1.00000,NaN,2005.000000,NaN,0.000000,NaN,0.000013,0.000010,NaN,NaN,...,NaN,0.000000,0.000000,0.000161,0.000000,NaN,NaN,30.000000,0.000000,0.100000
25%,23000.75000,NaN,2013.000000,NaN,856.000000,NaN,7.233425,6.725843,NaN,NaN,...,NaN,0.000000,0.000000,0.087777,1.000000,NaN,NaN,30.000000,0.000000,0.106637
50%,46000.50000,NaN,2020.000000,NaN,2070.000000,NaN,17.401696,16.245317,NaN,NaN,...,NaN,0.000000,0.000000,0.147868,1.000000,NaN,NaN,69.617815,0.000000,0.179359
75%,69000.25000,NaN,2021.000000,NaN,4146.000000,NaN,34.746243,32.681695,NaN,NaN,...,NaN,1.000000,1.000000,0.227352,1.000000,NaN,NaN,139.084008,3.775145,0.271323


In [5]:
# --- Vérification des types de données par colonne ---
print('Types de données par colonne :')
print(df.dtypes)


Types de données par colonne :
Identifiant_Véhicule        int64
Marque_et_Modèle              str
Année_Fabrication           int64
Type_Véhicule                 str
Heures_Utilisation          int64
Info_Itinéraire               str
Capacité_Charge           float64
Charge_Réelle             float64
Date_Dernier_Entretien        str
Type_Entretien                str
Coût_Entretien            float64
Température_Moteur        float64
Pression_Pneus            float64
Consommation_Carburant    float64
État_Batterie             float64
Niveaux_Vibration         float64
Qualité_Huile             float64
État_Freins                   str
Historique_Pannes           int64
Anomalies_Détectées         int64
Score_Prédictif           float64
Entretien_Nécessaire        int64
Conditions_Météo              str
Conditions_Route              str
Délais_Livraison          float64
Temps_Arrêt_Entretien     float64
Impact_Efficacité         float64
dtype: object


In [6]:
# --- Détection des valeurs manquantes ---
# Étape essentielle avant tout traitement pour identifier
# les colonnes nécessitant une imputation ou une suppression.
print('Nombre de valeurs nulles par colonne :')
print(df.isnull().sum())


Nombre de valeurs nulles par colonne :
Identifiant_Véhicule      0
Marque_et_Modèle          0
Année_Fabrication         0
Type_Véhicule             0
Heures_Utilisation        0
Info_Itinéraire           0
Capacité_Charge           0
Charge_Réelle             0
Date_Dernier_Entretien    0
Type_Entretien            0
Coût_Entretien            0
Température_Moteur        0
Pression_Pneus            0
Consommation_Carburant    0
État_Batterie             0
Niveaux_Vibration         0
Qualité_Huile             0
État_Freins               0
Historique_Pannes         0
Anomalies_Détectées       0
Score_Prédictif           0
Entretien_Nécessaire      0
Conditions_Météo          0
Conditions_Route          0
Délais_Livraison          0
Temps_Arrêt_Entretien     0
Impact_Efficacité         0
dtype: int64


## 3. Nettoyage des Données

Cette section couvre la suppression des doublons, le filtrage du périmètre d'étude
et le traitement des valeurs aberrantes (outliers).

In [7]:
# --- Suppression des lignes dupliquées ---
# Les doublons peuvent fausser les analyses et les modèles.
nb_avant = len(df)
df = df.drop_duplicates()
nb_apres = len(df)
print(f'{nb_avant - nb_apres} ligne(s) dupliquée(s) supprimée(s).')
print(f'Nombre de lignes restantes : {nb_apres}')


0 ligne(s) dupliquée(s) supprimée(s).
Nombre de lignes restantes : 92000


In [8]:
# --- Filtrage : sélection du modèle Volvo FH de type Camion ---
# Le périmètre d'étude se limite au modèle Volvo FH (Truck)
# afin d'obtenir un jeu de données homogène.

df1 = df[(df['Marque_et_Modèle'] == 'Volvo FH') & (df['Type_Véhicule'] == 'Truck')].copy()
df1.reset_index(drop=True, inplace=True)
df1['Identifiant_Véhicule'] = range(1, len(df1) + 1)

print(f'Nombre de lignes après filtrage : {len(df1)}')
df1[['Identifiant_Véhicule', 'Marque_et_Modèle', 'Type_Véhicule']].head()


Nombre de lignes après filtrage : 4092


,Identifiant_Véhicule,Marque_et_Modèle,Type_Véhicule
0,1,Volvo FH,Truck
1,2,Volvo FH,Truck
2,3,Volvo FH,Truck
3,4,Volvo FH,Truck
4,5,Volvo FH,Truck


In [9]:
# --- Suppression des colonnes non pertinentes pour l'analyse ---
# Les colonnes suivantes sont hors périmètre ou redondantes :
# Marque_et_Modèle, Type_Véhicule, Délais_Livraison,
# Temps_Arrêt_Entretien, Impact_Efficacité, Coût_Entretien.

colonnes_a_supprimer = [
    'Marque_et_Modèle', 'Type_Véhicule',
    'Délais_Livraison', 'Temps_Arrêt_Entretien',
    'Impact_Efficacité', 'Coût_Entretien'
]
df1.drop(colonnes_a_supprimer, axis=1, inplace=True)
print(f'Colonnes conservées : {list(df1.columns)}')
df1.head()


Colonnes conservées : ['Identifiant_Véhicule', 'Année_Fabrication', 'Heures_Utilisation', 'Info_Itinéraire', 'Capacité_Charge', 'Charge_Réelle', 'Date_Dernier_Entretien', 'Type_Entretien', 'Température_Moteur', 'Pression_Pneus', 'Consommation_Carburant', 'État_Batterie', 'Niveaux_Vibration', 'Qualité_Huile', 'État_Freins', 'Historique_Pannes', 'Anomalies_Détectées', 'Score_Prédictif', 'Entretien_Nécessaire', 'Conditions_Météo', 'Conditions_Route']


,Identifiant_Véhicule,Année_Fabrication,Heures_Utilisation,Info_Itinéraire,Capacité_Charge,Charge_Réelle,Date_Dernier_Entretien,Type_Entretien,Température_Moteur,Pression_Pneus,...,État_Batterie,Niveaux_Vibration,Qualité_Huile,État_Freins,Historique_Pannes,Anomalies_Détectées,Score_Prédictif,Entretien_Nécessaire,Conditions_Météo,Conditions_Route
0,1,2020,4506,Highway,62.037153,45.324843,15/05/2024,Oil Change,120.0,55.0,...,45.0,1.355190,81.826548,Fair,0,0,0.199099,0,Clear,Highway
1,2,2020,3615,Highway,5.969894,6.383313,09/07/2023,Engine Overhaul,120.0,55.0,...,45.0,4.286082,84.369240,Good,0,1,0.149013,1,Clear,Urban
2,3,2021,128,Urban,41.784895,47.036831,03/02/2023,Tire Rotation,120.0,55.0,...,45.0,4.629584,89.845122,Poor,0,0,0.251046,1,Clear,Rural
3,4,2016,6379,Highway,2.909616,2.560338,08/04/2023,Oil Change,120.0,20.0,...,45.0,2.106645,91.277795,Good,1,1,0.117144,1,Clear,Rural
4,5,2005,44,Urban,8.864533,6.675255,13/03/2023,Tire Rotation,120.0,20.0,...,45.0,1.455468,71.813017,Poor,1,0,0.469806,1,Clear,Urban


In [10]:
# --- Analyse statistique de la variable Capacité_Charge ---
# Permet d'identifier les valeurs aberrantes ou incohérentes.

print('=== Statistiques descriptives de Capacité_Charge ===')
print(df1['Capacité_Charge'].describe())


=== Statistiques descriptives de Capacité_Charge ===
count    4092.000000
mean       25.495392
std        25.688815
min         0.017976
25%         7.333189
50%        17.736672
75%        35.350269
max       255.985201
Name: Capacité_Charge, dtype: float64


In [11]:
# --- Détection d'une valeur suspecte dans Capacité_Charge ---
# Vérification si une valeur précise (ex. 255.985201) est sur-représentée,
# ce qui pourrait indiquer une erreur de saisie ou de génération.

valeur_cible = 255.985201
tolerance = 1e-6

nb_exact  = (df1['Capacité_Charge'] == valeur_cible).sum()
nb_approx = ((df1['Capacité_Charge'] - valeur_cible).abs() < tolerance).sum()

print(f'Correspondances exactes  : {nb_exact}')
print(f'Correspondances (tol={tolerance}) : {nb_approx}')

lignes_trouvees = df1[(df1['Capacité_Charge'] - valeur_cible).abs() < tolerance]
print('\nAperçu des lignes concernées :')
print(lignes_trouvees[['Capacité_Charge']].head())


Correspondances exactes  : 0
Correspondances (tol=1e-06) : 1

Aperçu des lignes concernées :
     Capacité_Charge
633       255.985201


In [12]:
# --- Identification des valeurs hors plage réaliste ---
# Un camion Volvo FH a une capacité maximale standard d'environ 26 à 30 tonnes.
# Toute valeur largement supérieure est considérée comme aberrante.

nb_capacite = (df1['Capacité_Charge'] > 50).sum()
nb_charge   = (df1['Charge_Réelle']   > 60).sum()

print(f'Lignes avec Capacité_Charge > 50 tonnes : {nb_capacite}')
print(f'Lignes avec Charge_Réelle   > 60 tonnes : {nb_charge}')


Lignes avec Capacité_Charge > 50 tonnes : 581
Lignes avec Charge_Réelle   > 60 tonnes : 346


In [13]:
# --- Suppression des observations aberrantes (Capacité_Charge > 55 tonnes) ---
# Seules les lignes dont la capacité de charge est réaliste sont conservées.

df1 = df1[df1['Capacité_Charge'] <= 55].copy()
df1.reset_index(drop=True, inplace=True)
df1['Identifiant_Véhicule'] = range(1, len(df1) + 1)

print(f'Nombre de lignes après filtrage : {len(df1)}')
print('\nVérification des premières lignes :')
print(df1[['Identifiant_Véhicule', 'Capacité_Charge']].head())
print('\nVérification des dernières lignes :')
print(df1[['Identifiant_Véhicule', 'Capacité_Charge']].tail())


Nombre de lignes après filtrage : 3618

Vérification des premières lignes :
   Identifiant_Véhicule  Capacité_Charge
0                     1         5.969894
1                     2        41.784895
2                     3         2.909616
3                     4         8.864533
4                     5        21.902258

Vérification des dernières lignes :
      Identifiant_Véhicule  Capacité_Charge
3613                  3614        48.206528
3614                  3615        22.463914
3615                  3616         8.239716
3616                  3617         7.371252
3617                  3618        16.798042


In [14]:
# --- Vérification de la variable Charge_Réelle après nettoyage ---

nb_charge = (df1['Charge_Réelle'] > 55).sum()
print(f'Nombre de lignes avec Charge_Réelle > 55 tonnes : {nb_charge}')


Nombre de lignes avec Charge_Réelle > 55 tonnes : 34


## 4. Standardisation des Variables Catégorielles

Traduction des modalités anglaises en français et normalisation de la casse
pour garantir la cohérence du jeu de données.

In [15]:
# --- Traduction des valeurs textuelles (anglais -> français) ---
# Standardisation des modalités pour les variables catégorielles.

df1['Info_Itinéraire'] = df1['Info_Itinéraire'].replace({
    'Rural'  : 'rural',
    'Highway': 'autoroute',
    'Urban'  : 'urbain'
})

df1['Conditions_Route'] = df1['Conditions_Route'].replace({
    'Rural'  : 'rural',
    'Highway': 'autoroute',
    'Urban'  : 'urbain'
})

df1['Type_Entretien'] = df1['Type_Entretien'].replace({
    'Engine Overhaul': 'révision du moteur',
    'Oil Change'     : "changement d'huile",
    'Tire Rotation'  : 'rotation des pneus',
})

df1['Conditions_Météo'] = df1['Conditions_Météo'].replace({
    'Clear' : 'clair',
    'Rainy' : 'pluvieux',
    'Snowy' : 'neigeux',
    'Windy' : 'venteux'
})

df1['État_Freins'] = df1['État_Freins'].replace({
    'Good': 'bon',
    'Fair': 'moyen',
    'Poor': 'mauvais'
})

# Vérification des valeurs uniques après traduction
for col in ['Info_Itinéraire', 'Conditions_Route', 'Type_Entretien', 'Conditions_Météo', 'État_Freins']:
    print(f'{col} : {df1[col].unique()}')


Info_Itinéraire : <StringArray>
['autoroute', 'urbain', 'rural']
Length: 3, dtype: str
Conditions_Route : <StringArray>
['urbain', 'rural', 'autoroute']
Length: 3, dtype: str
Type_Entretien : <StringArray>
['révision du moteur', 'rotation des pneus', 'changement d'huile']
Length: 3, dtype: str
Conditions_Météo : <StringArray>
['clair', 'pluvieux', 'venteux', 'neigeux']
Length: 4, dtype: str
État_Freins : <StringArray>
['bon', 'mauvais', 'moyen']
Length: 3, dtype: str


In [16]:
# --- Normalisation des chaînes de caractères (mise en minuscules) ---
# Garantit l'uniformité des modalités textuelles
# et évite les incohérences dues à la casse.

colonnes_texte = df1.select_dtypes(include=['object']).columns
df1[colonnes_texte] = df1[colonnes_texte].apply(lambda x: x.str.lower())

print('Aperçu des colonnes textuelles après normalisation :')
df1[colonnes_texte].head()


Aperçu des colonnes textuelles après normalisation :


,Info_Itinéraire,Date_Dernier_Entretien,Type_Entretien,État_Freins,Conditions_Météo,Conditions_Route
0,autoroute,09/07/2023,révision du moteur,bon,clair,urbain
1,urbain,03/02/2023,rotation des pneus,mauvais,clair,rural
2,autoroute,08/04/2023,changement d'huile,bon,clair,rural
3,urbain,13/03/2023,rotation des pneus,mauvais,clair,urbain
4,rural,09/05/2023,changement d'huile,bon,clair,autoroute


In [17]:
# --- Conversion de la colonne de date en type datetime ---
# Nécessaire pour tout calcul ou filtrage temporel ultérieur.

df1['Date_Dernier_Entretien'] = pd.to_datetime(
    df1['Date_Dernier_Entretien'], errors='coerce'
)

print('Type de la colonne après conversion :', df1['Date_Dernier_Entretien'].dtype)
df1['Date_Dernier_Entretien'].head()


Type de la colonne après conversion : datetime64[us]


0   2023-09-07
1   2023-03-02
2   2023-08-04
3          NaT
4   2023-09-05
Name: Date_Dernier_Entretien, dtype: datetime64[us]

In [18]:
# --- Vérification de la cohérence entre Conditions_Route et Info_Itinéraire ---
# Ces deux colonnes devraient contenir les mêmes informations.
# Un écart signifie une incohérence dans la source de données.

comparaison = df1['Conditions_Route'] == df1['Info_Itinéraire']
nb_differences = (~comparaison).sum()

print(f'Nombre de lignes divergentes : {nb_differences}')
if nb_differences > 0:
    print('Exemples de divergences :')
    print(df1[~comparaison][['Conditions_Route', 'Info_Itinéraire']].head())
else:
    print('Les deux colonnes sont parfaitement identiques sur tout le jeu de données.')

print(f'Taux de concordance : {comparaison.mean() * 100:.2f}%')


Nombre de lignes divergentes : 2345
Exemples de divergences :
  Conditions_Route Info_Itinéraire
0           urbain       autoroute
1            rural          urbain
2            rural       autoroute
4        autoroute           rural
6            rural          urbain
Taux de concordance : 35.19%


## 5. Génération Synthétique des Variables Capteurs

Les variables suivantes sont recalculées à partir de modèles paramétriques
basés sur les caractéristiques physiques du camion Volvo FH :
- Température moteur (°C)
- Pression des pneus (kPa)
- Consommation carburant (L/100 km)
- Etat de la batterie (%)
- Niveaux de vibration

In [19]:
# ─────────────────────────────────────────────────────────────────
# Génération synthétique de la variable Température_Moteur
# Modèle basé sur : surcharge, anomalies, pannes, météo,
#                   conditions routières, heures d'utilisation.
# Plage réaliste pour un Volvo FH : 75 à 128 °C
# ─────────────────────────────────────────────────────────────────

def generer_temperature_moteur(ligne):
    """
    Estime la température moteur (°C) d'un camion Volvo FH
    en fonction de plusieurs paramètres opérationnels.
    """
    temperature = random.uniform(82, 96)  # plage normale de fonctionnement

    # Effet de la surcharge
    overload_ratio = ligne['Charge_Réelle'] / ligne['Capacité_Charge']
    if overload_ratio > 1:
        temperature += min((overload_ratio - 1) * 25, 18)

    # Anomalies détectées
    if ligne['Anomalies_Détectées'] == 1:
        temperature += random.uniform(8, 16)

    # Historique de pannes
    if ligne['Historique_Pannes'] == 1:
        temperature += random.uniform(3, 7)

    # Influence de la météo
    if   ligne['Conditions_Météo'] == 'pluvieux':
        temperature -= random.uniform(2, 4)
    elif ligne['Conditions_Météo'] == 'clair':
        temperature += random.uniform(4, 6)
    elif ligne['Conditions_Météo'] == 'neigeux':
        temperature -= random.uniform(3, 6)

    # Influence du type de route
    if   ligne['Conditions_Route'] == 'urbain':
        temperature += random.uniform(2, 5)   # arrêts fréquents
    elif ligne['Conditions_Route'] == 'autoroute':
        temperature += random.uniform(5, 10)  # vitesse soutenue

    # Dégradation liée aux heures d'utilisation
    temperature += min(ligne['Heures_Utilisation'] / 5000, 9)

    # Bruit de capteur
    temperature += random.uniform(-1.5, 1.5)

    # Encadrement dans la plage réaliste
    temperature = min(max(temperature, 75), 128)
    return round(temperature, 1)


df1['Température_Moteur'] = df1.apply(generer_temperature_moteur, axis=1)
print('Aperçu des températures moteur générées :')
print(df1[['Identifiant_Véhicule', 'Température_Moteur']].head())


Aperçu des températures moteur générées :
   Identifiant_Véhicule  Température_Moteur
0                     1               112.5
1                     2                96.7
2                     3               107.1
3                     4               108.2
4                     5               121.4


In [20]:
# ─────────────────────────────────────────────────────────────────
# Génération synthétique de la variable Pression_Pneus
# Modèle basé sur : surcharge, météo, anomalies, heures d'utilisation.
# Plage réaliste pour un camion lourd : 85 à 140 kPa
# ─────────────────────────────────────────────────────────────────

def generer_pression_pneus(ligne):
    """
    Estime la pression des pneus (kPa) d'un Volvo FH
    selon les conditions d'exploitation.
    """
    pression = random.uniform(112, 125)  # pression nominale

    # Surcharge mécanique sur les pneus
    overload_ratio = ligne['Charge_Réelle'] / ligne['Capacité_Charge']
    if overload_ratio > 1:
        pression += min((overload_ratio - 1) * 10, 8)

    # Influence de la météo
    if   ligne['Conditions_Météo'] == 'clair':
        pression += random.uniform(2, 5)
    elif ligne['Conditions_Météo'] == 'neigeux':
        pression -= random.uniform(1, 3)

    # Anomalies : fuite d'air possible
    if ligne['Anomalies_Détectées'] == 1:
        if random.random() < 0.6:
            pression -= random.uniform(8, 18)

    # Vieillissement des pneus
    pression -= min(ligne['Heures_Utilisation'] / 10000, 4)

    # Bruit de capteur
    pression += random.uniform(-1.5, 1.5)

    pression = min(max(pression, 85), 140)
    return round(pression, 1)


df1['Pression_Pneus'] = df1.apply(generer_pression_pneus, axis=1)
print('Aperçu des pressions de pneus générées :')
print(df1[['Identifiant_Véhicule', 'Pression_Pneus']].head())


Aperçu des pressions de pneus générées :
   Identifiant_Véhicule  Pression_Pneus
0                     1           117.1
1                     2           115.9
2                     3           117.1
3                     4           127.2
4                     5           123.6


In [21]:
# ─────────────────────────────────────────────────────────────────
# Génération synthétique de la variable Consommation_Carburant
# Modèle basé sur : charge, type de route, météo,
#                   historique de pannes, heures d'utilisation.
# Plage réaliste : 24 à 55 L/100 km
# ─────────────────────────────────────────────────────────────────

def generer_consommation_carburant(ligne):
    """
    Estime la consommation de carburant (L/100 km) du Volvo FH
    en fonction des conditions d'utilisation.
    """
    consommation = random.uniform(29, 36)  # consommation de base

    # Effet de la charge transportée
    overload_ratio = ligne['Charge_Réelle'] / ligne['Capacité_Charge']
    consommation += min(overload_ratio * 6, 12)

    # Influence du type de route
    route = ligne['Conditions_Route']
    if   route == 'urbain':
        consommation += random.uniform(3, 6)   # arrêts fréquents
    elif route == 'rural':
        consommation += random.uniform(5, 10)  # relief et sinuosités
    elif route == 'autoroute':
        consommation -= random.uniform(1, 3)   # allure stable

    # Influence météorologique
    meteo = ligne['Conditions_Météo']
    if   meteo == 'neigeux':
        consommation += random.uniform(2, 5)
    elif meteo == 'pluvieux':
        consommation += random.uniform(1, 3)
    elif meteo == 'clair':
        consommation += random.uniform(0.5, 1.5)

    # Dégradation moteur liée aux pannes
    if ligne['Historique_Pannes'] == 1:
        consommation += random.uniform(2, 5)

    # Vieillissement mécanique
    consommation += min(ligne['Heures_Utilisation'] / 7000, 6)

    # Défauts détectés
    if ligne['Anomalies_Détectées'] == 1:
        consommation += random.uniform(3, 8)

    # Bruit de mesure
    consommation += random.uniform(-1.5, 1.5)

    consommation = min(max(consommation, 24), 55)
    return round(consommation, 1)


df1['Consommation_Carburant'] = df1.apply(generer_consommation_carburant, axis=1)
print('Aperçu des consommations de carburant générées :')
print(df1[['Identifiant_Véhicule', 'Consommation_Carburant']].head())


Aperçu des consommations de carburant générées :
   Identifiant_Véhicule  Consommation_Carburant
0                     1                    45.0
1                     2                    47.5
2                     3                    55.0
3                     4                    46.9
4                     5                    45.7


In [22]:
# ─────────────────────────────────────────────────────────────────
# Génération synthétique de la variable Etat_Batterie
# Modèle basé sur : heures d'utilisation, pannes, anomalies,
#                   météo et conditions routières.
# Plage réaliste : 15 à 100 %
# ─────────────────────────────────────────────────────────────────

def generer_etat_batterie(ligne):
    """
    Estime le niveau de charge de la batterie (%) d'un Volvo FH
    selon le vieillissement et les conditions d'exploitation.
    """
    batterie = random.uniform(80, 100)  # état initial nominal

    # Dégradation progressive avec l'usage
    batterie -= min(ligne['Heures_Utilisation'] / 2500, 35)

    # Impact des pannes historiques
    if ligne['Historique_Pannes'] == 1:
        batterie -= random.uniform(5, 12)

    # Impact des anomalies électriques
    if ligne['Anomalies_Détectées'] == 1:
        batterie -= random.uniform(8, 18)

    # Conditions météorologiques
    meteo = ligne['Conditions_Météo']
    if   meteo == 'neigeux':
        batterie -= random.uniform(4, 8)
    elif meteo == 'pluvieux':
        batterie -= random.uniform(2, 5)
    elif meteo == 'venteux':
        batterie -= random.uniform(1, 3)

    # Conditions routières
    route = ligne['Conditions_Route']
    if   route == 'urbain':
        batterie -= random.uniform(2, 5)  # sollicitations fréquentes
    elif route == 'rural':
        batterie -= random.uniform(1, 3)

    # Bruit de capteur
    batterie += random.uniform(-2, 2)

    batterie = min(max(batterie, 15), 100)
    return round(batterie, 1)


df1['Etat_Batterie'] = df1.apply(generer_etat_batterie, axis=1)
print('Aperçu de l\'état de la batterie généré :')
print(df1[['Identifiant_Véhicule', 'Etat_Batterie']].head())


Aperçu de l'état de la batterie généré :
   Identifiant_Véhicule  Etat_Batterie
0                     1           63.6
1                     2           84.1
2                     3           75.1
3                     4           80.4
4                     5           75.8


In [23]:
# ─────────────────────────────────────────────────────────────────
# Génération synthétique de la variable Niveaux_Vibration
# Modèle basé sur : usure mécanique, anomalies, pannes,
#                   état des freins, qualité de l'huile,
#                   conditions routières et météo.
# Plage réaliste : 1 à 12 (unité arbitraire de vibration)
# ─────────────────────────────────────────────────────────────────

def generer_niveaux_vibration(ligne):
    """
    Estime le niveau de vibration d'un camion Volvo FH
    en fonction de l'état mécanique et des conditions d'exploitation.
    """
    vibration = random.uniform(1.8, 4.2)  # niveau de base nominal

    # Dégradation mécanique liée à l'usage
    vibration += min(ligne['Heures_Utilisation'] / 9000, 4)

    # Anomalies mécaniques
    if ligne['Anomalies_Détectées'] == 1:
        vibration += random.uniform(2, 5)

    # Historique de pannes
    if ligne['Historique_Pannes'] == 1:
        vibration += random.uniform(1, 3)

    # État des freins
    if str(ligne['État_Freins']).lower() in ['mauvais', 'critique', 'usé']:
        vibration += random.uniform(1.5, 3.5)

    # Qualité de l'huile moteur
    if float(ligne['Qualité_Huile']) < 40:
        vibration += random.uniform(1, 3)

    # Type de route
    route = ligne['Conditions_Route']
    if   route == 'urbain':
        vibration += random.uniform(0.5, 1.5)
    elif route == 'rural':
        vibration += random.uniform(1, 2)
    elif route == 'autoroute':
        vibration -= random.uniform(0.2, 0.8)  # roulage stable

    # Conditions météorologiques
    meteo = ligne['Conditions_Météo']
    if   meteo == 'neigeux':
        vibration += random.uniform(0.5, 1.5)
    elif meteo == 'venteux':
        vibration += random.uniform(0.3, 1)

    # Bruit de capteur
    vibration += random.uniform(-0.3, 0.3)

    vibration = min(max(vibration, 1), 12)
    return round(vibration, 2)


df1['Niveaux_Vibration'] = df1.apply(generer_niveaux_vibration, axis=1)
print('Aperçu des niveaux de vibration générés :')
print(df1[['Identifiant_Véhicule', 'Niveaux_Vibration']].head())


Aperçu des niveaux de vibration générés :
   Identifiant_Véhicule  Niveaux_Vibration
0                     1               6.65
1                     2               7.19
2                     3               9.19
3                     4               8.51
4                     5               7.71


In [24]:
"""
====================================================================
  Score Prédictif de Maintenance — Fonction de calcul raisonnée
====================================================================

LOGIQUE GÉNÉRALE
----------------
Le score final est calculé en 3 étapes :

  1. Calcul de 7 sous-scores partiels, chacun sur [0, 1]
     (0 = parfait état, 1 = état critique)

  2. Agrégation pondérée → score brut sur [0, 1]

  3. Mise à l'échelle par Power Transform (exposant 0.5)
     après Min-Max sur les bornes réelles observées.
     Cela étale le score sur tout [0, 1] sans déformer
     le classement, et pénalise davantage les cas extrêmes.

POURQUOI PAS LA SIGMOÏDE ?
---------------------------
La sigmoïde compresse les queues et laisse le 75e centile
autour de 0.42. Le Power Transform (racine carrée) étale
naturellement les valeurs basses tout en conservant les
valeurs hautes proches de 1. Résultat :
  - 75e centile ≈ 0.72   ✓
  - Cas critique → proche de 1.0  ✓
  - Cas sain     → proche de 0.0  ✓

SOUS-SCORES ET POIDS
---------------------
  Composant              Poids   Justification
  ---------------------  ------  -----------------------------------------
  Température moteur      0.20   Surchauffe → dommages immédiats
  État freins             0.20   Sécurité critique
  Niveaux vibration       0.15   Indique usure mécanique
  Qualité huile           0.15   Lubrifiant dégradé → pannes moteur
  État batterie           0.10   Alimentation électronique
  Heures utilisation      0.10   Vieillissement progressif (courbe √)
  Anomalies + Pannes      0.10   Historique de défaillances
  ---------------------  ------  -----------------------------------------
  TOTAL                   1.00

BORNES DE NORMALISATION (calibrées sur le dataset réel)
---------------------------------------------------------
  SCORE_BRUT_MIN = 0.0064   (meilleur véhicule observé)
  SCORE_BRUT_MAX = 0.8647   (pire véhicule observé)

  Ces valeurs peuvent être recalibrées via calibrer_bornes()
  si le dataset évolue.

SEUILS DE RISQUE (score final)
--------------------------------
  [0.00 – 0.30[  →  Faible
  [0.30 – 0.55[  →  Modéré
  [0.55 – 0.75[  →  Élevé
  [0.75 – 1.00]  →  Critique
====================================================================
"""

import pandas as pd
import numpy as np


# ──────────────────────────────────────────────────────────────────
#  BORNES DE CALIBRATION
#  Ajustez ces valeurs si vous recalibrez sur un nouveau dataset.
# ──────────────────────────────────────────────────────────────────

SCORE_BRUT_MIN: float = 0.0064   # score brut du meilleur véhicule observé
SCORE_BRUT_MAX: float = 0.8647   # score brut du pire véhicule observé
POWER_EXPOSANT: float = 0.5      # exposant du Power Transform (0.5 = racine carrée)


# ──────────────────────────────────────────────────────────────────
#  POIDS DES COMPOSANTS
# ──────────────────────────────────────────────────────────────────

POIDS: dict[str, float] = {
    "temperature" : 0.20,
    "freins"      : 0.20,
    "vibration"   : 0.15,
    "huile"       : 0.15,
    "batterie"    : 0.10,
    "heures"      : 0.10,
    "historique"  : 0.10,
}

assert abs(sum(POIDS.values()) - 1.0) < 1e-9, "Les poids doivent sommer à 1.0"


# ──────────────────────────────────────────────────────────────────
#  UTILITAIRES
# ──────────────────────────────────────────────────────────────────

def clamp(value: float, low: float, high: float) -> float:
    """Ramène une valeur dans l'intervalle [low, high]."""
    return max(low, min(high, value))


def power_transform(score_brut: float) -> float:
    """
    Transforme le score brut [0,1] en score final [0,1]
    via un Min-Max sur les bornes réelles + Power Transform.

    Étape 1 : Min-Max sur [SCORE_BRUT_MIN, SCORE_BRUT_MAX]
    Étape 2 : x^POWER_EXPOSANT  (racine carrée par défaut)

    La racine carrée étale les valeurs basses (la majorité)
    et laisse les valeurs critiques proches de 1.
    """
    normalise = (score_brut - SCORE_BRUT_MIN) / (SCORE_BRUT_MAX - SCORE_BRUT_MIN)
    normalise = clamp(normalise, 0.0, 1.0)
    return round(normalise ** POWER_EXPOSANT, 4)


# ──────────────────────────────────────────────────────────────────
#  SOUS-SCORES PARTIELS
# ──────────────────────────────────────────────────────────────────

def score_temperature(temp: float) -> float:
    """
    Température moteur (°C) — zone normale : 80-95°C
      < 95°C       → 0.0  (normal)
      95  – 110°C  → 0.0 – 0.40  (attention, linéaire)
      110 – 120°C  → 0.40 – 1.0  (critique, accéléré)
      > 120°C      → 1.0  (surchauffe)
    """
    if temp < 95:
        return 0.0
    elif temp <= 110:
        return clamp((temp - 95) / 15 * 0.4, 0, 1)
    elif temp <= 120:
        return clamp(0.4 + (temp - 110) / 10 * 0.6, 0, 1)
    else:
        return 1.0


def score_vibration(vib: float) -> float:
    """
    Niveau de vibration [1 – 12]
      < 3    → 0.0  (silencieux)
      3 – 7  → 0.0 – 0.50  (modéré)
      7 – 10 → 0.50 – 1.0  (élevé, accéléré)
      > 10   → 1.0  (critique)
    """
    if vib < 3:
        return 0.0
    elif vib <= 7:
        return clamp((vib - 3) / 4 * 0.5, 0, 1)
    elif vib <= 10:
        return clamp(0.5 + (vib - 7) / 3 * 0.5, 0, 1)
    else:
        return 1.0


def score_qualite_huile(qualite: float) -> float:
    """
    Qualité de l'huile [0 – 100] (100 = parfait)
      > 85  → 0.0   (excellent)
      85-65 → 0.0 – 0.40  (normal)
      65-50 → 0.40 – 1.0  (dégradée, accéléré)
      < 50  → 1.0   (critique)
    """
    if qualite > 85:
        return 0.0
    elif qualite >= 65:
        return clamp((85 - qualite) / 20 * 0.4, 0, 1)
    elif qualite >= 50:
        return clamp(0.4 + (65 - qualite) / 15 * 0.6, 0, 1)
    else:
        return 1.0


def score_freins(etat: str) -> float:
    """
    État des freins (catégorielle)
      'bon'     → 0.0
      'moyen'   → 0.5
      'mauvais' → 1.0
    """
    mapping = {"bon": 0.0, "moyen": 0.5, "mauvais": 1.0}
    return mapping.get(str(etat).strip().lower(), 0.5)


def score_batterie(batterie: float) -> float:
    """
    État batterie [0 – 100] (100 = parfait)
      > 80  → 0.0   (bon)
      80-60 → 0.0 – 0.40  (normal)
      60-45 → 0.40 – 1.0  (faible, accéléré)
      < 45  → 1.0   (critique)
    """
    if batterie > 80:
        return 0.0
    elif batterie >= 60:
        return clamp((80 - batterie) / 20 * 0.4, 0, 1)
    elif batterie >= 45:
        return clamp(0.4 + (60 - batterie) / 15 * 0.6, 0, 1)
    else:
        return 1.0


def score_heures(heures: float, max_heures: float = 28638) -> float:
    """
    Heures d'utilisation — vieillissement progressif.
    Courbe √ : l'usure accélère avec le temps.
    max_heures = maximum observé dans le dataset (28 638 h).
    """
    return clamp(np.sqrt(heures / max_heures), 0, 1)


def score_historique(pannes: int, anomalies: int) -> float:
    """
    Historique de défaillances (0 ou 1 chacun)
    Pannes pondérées davantage (60%) car plus sérieuses.
    """
    return clamp(0.6 * pannes + 0.4 * anomalies, 0, 1)


# ──────────────────────────────────────────────────────────────────
#  FONCTION PRINCIPALE
# ──────────────────────────────────────────────────────────────────

def calculer_score_predictif(
    temperature_moteur:   float,
    niveaux_vibration:    float,
    qualite_huile:        float,
    etat_freins:          str,
    etat_batterie:        float,
    heures_utilisation:   float,
    historique_pannes:    int,
    anomalies_detectees:  int,
) -> dict:
    """
    Calcule le Score Prédictif de Maintenance d'un véhicule.

    Paramètres
    ----------
    temperature_moteur   : float — Température moteur en °C
    niveaux_vibration    : float — Niveau de vibration (1–12)
    qualite_huile        : float — Qualité de l'huile (0–100)
    etat_freins          : str   — 'bon' | 'moyen' | 'mauvais'
    etat_batterie        : float — État batterie (0–100)
    heures_utilisation   : float — Heures totales d'utilisation
    historique_pannes    : int   — 0 ou 1
    anomalies_detectees  : int   — 0 ou 1

    Retourne
    --------
    dict :
      'score'   : float [0,1]   Score final (0 = sain, 1 = critique)
      'niveau'  : str           'Faible' | 'Modéré' | 'Élevé' | 'Critique'
      'details' : dict          Sous-scores par composant [0,1]
    """

    # ── 1. Sous-scores ──
    details = {
        "temperature" : score_temperature(temperature_moteur),
        "vibration"   : score_vibration(niveaux_vibration),
        "huile"       : score_qualite_huile(qualite_huile),
        "freins"      : score_freins(etat_freins),
        "batterie"    : score_batterie(etat_batterie),
        "heures"      : score_heures(heures_utilisation),
        "historique"  : score_historique(historique_pannes, anomalies_detectees),
    }

    # ── 2. Score brut pondéré ──
    score_brut = sum(POIDS[k] * v for k, v in details.items())

    # ── 3. Power Transform → score final [0,1] ──
    score_final = power_transform(score_brut)

    # ── 4. Niveau de risque ──
    if score_final < 0.30:
        niveau = "Faible"
    elif score_final < 0.55:
        niveau = "Modéré"
    elif score_final < 0.75:
        niveau = "Élevé"
    else:
        niveau = "Critique"

    return {
        "score"  : score_final,
        "niveau" : niveau,
        "details": {k: round(v, 4) for k, v in details.items()},
    }


# ──────────────────────────────────────────────────────────────────
#  CALIBRATION (optionnelle — à relancer si le dataset change)
# ──────────────────────────────────────────────────────────────────

def calibrer_bornes(df: pd.DataFrame) -> tuple[float, float]:
    """
    Recalcule SCORE_BRUT_MIN et SCORE_BRUT_MAX sur un DataFrame.
    À utiliser si le dataset évolue significativement.

    Retourne (min_brut, max_brut) à renseigner dans les constantes.
    """
    scores = []
    for _, row in df.iterrows():
        d = {
            "temperature" : score_temperature(row["Température_Moteur"]),
            "vibration"   : score_vibration(row["Niveaux_Vibration"]),
            "huile"       : score_qualite_huile(row["Qualité_Huile"]),
            "freins"      : score_freins(row["État_Freins"]),
            "batterie"    : score_batterie(row["Etat_Batterie"]),
            "heures"      : score_heures(row["Heures_Utilisation"]),
            "historique"  : score_historique(row["Historique_Pannes"], row["Anomalies_Détectées"]),
        }
        scores.append(sum(POIDS[k] * v for k, v in d.items()))
    return min(scores), max(scores)


# ──────────────────────────────────────────────────────────────────
#  APPLICATION SUR DATAFRAME
# ──────────────────────────────────────────────────────────────────

def appliquer_sur_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applique calculer_score_predictif sur chaque ligne et retourne
    un DataFrame avec les colonnes Score_Prédictif et Niveau_Risque
    mises à jour.
    """
    resultats = df.apply(
        lambda row: calculer_score_predictif(
            temperature_moteur  = row["Température_Moteur"],
            niveaux_vibration   = row["Niveaux_Vibration"],
            qualite_huile       = row["Qualité_Huile"],
            etat_freins         = row["État_Freins"],
            etat_batterie       = row["Etat_Batterie"],
            heures_utilisation  = row["Heures_Utilisation"],
            historique_pannes   = row["Historique_Pannes"],
            anomalies_detectees = row["Anomalies_Détectées"],
        ),
        axis=1,
        result_type="expand",
    )
    df = df.copy()
    df["Score_Prédictif"] = resultats["score"]
    df["Niveau_Risque"]   = resultats["niveau"]
    return df


# ──────────────────────────────────────────────────────────────────
#  POINT D'ENTRÉE
# ──────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # ── Test unitaire ──────────────────────────────────────────────
    print("=" * 58)
    print("  TEST UNITAIRE — Véhicule en mauvais état")
    print("=" * 58)

    res = calculer_score_predictif(
        temperature_moteur  = 118.0,
        niveaux_vibration   = 9.5,
        qualite_huile       = 58.0,
        etat_freins         = "mauvais",
        etat_batterie       = 52.0,
        heures_utilisation  = 18000,
        historique_pannes   = 1,
        anomalies_detectees = 1,
    )
    print(f"\n  Score Final   : {res['score']}")
    print(f"  Niveau Risque : {res['niveau']}")
    print("\n  Détail sous-scores :")
    for comp, val in res["details"].items():
        barre = "█" * int(val * 20) + "░" * (20 - int(val * 20))
        print(f"    {comp:<13} [{barre}] {val:.2f}")

    print("\n" + "=" * 58)
    print("  TEST UNITAIRE — Véhicule en bon état")
    print("=" * 58)

    res2 = calculer_score_predictif(
        temperature_moteur  = 92.0,
        niveaux_vibration   = 2.1,
        qualite_huile       = 91.0,
        etat_freins         = "bon",
        etat_batterie       = 88.0,
        heures_utilisation  = 500,
        historique_pannes   = 0,
        anomalies_detectees = 0,
    )
    print(f"\n  Score Final   : {res2['score']}")
    print(f"  Niveau Risque : {res2['niveau']}")

    # ── Application sur le dataset ─────────────────────────────────
    print("\n" + "=" * 58)
    print("  APPLICATION SUR LE DATASET COMPLET")
    print("=" * 58)

    df1 = appliquer_sur_dataframe(df1)

    print(f"\nStatistiques Score_Prédictif (nouveau) :")
    print(df1["Score_Prédictif"].describe().round(4))

    print(f"\nPercentiles :")
    for p in [10, 25, 50, 75, 90, 95, 99]:
        print(f"  {p}% : {df1['Score_Prédictif'].quantile(p/100):.4f}")

    print(f"\nDistribution Niveau_Risque :")
    print(df1["Niveau_Risque"].value_counts())

    print(f"\nScore moyen par Entretien_Nécessaire :")
    print(df1.groupby("Entretien_Nécessaire")["Score_Prédictif"].mean().round(4))

  TEST UNITAIRE — Véhicule en mauvais état

  Score Final   : 1.0
  Niveau Risque : Critique

  Détail sous-scores :
    temperature   [█████████████████░░░] 0.88
    vibration     [██████████████████░░] 0.92
    huile         [█████████████░░░░░░░] 0.68
    freins        [████████████████████] 1.00
    batterie      [██████████████░░░░░░] 0.72
    heures        [███████████████░░░░░] 0.79
    historique    [████████████████████] 1.00

  TEST UNITAIRE — Véhicule en bon état

  Score Final   : 0.0891
  Niveau Risque : Faible

  APPLICATION SUR LE DATASET COMPLET

Statistiques Score_Prédictif (nouveau) :
count    3618.0000
mean        0.5911
std         0.1695
min         0.0451
25%         0.4785
50%         0.6036
75%         0.7222
max         0.9973
Name: Score_Prédictif, dtype: float64

Percentiles :
  10% : 0.3486
  25% : 0.4785
  50% : 0.6036
  75% : 0.7221
  90% : 0.7982
  95% : 0.8482
  99% : 0.9170

Distribution Niveau_Risque :
Niveau_Risque
Élevé       1570
Modéré      1137
Cr

## 6. Vérification Finale et Export

Contrôle visuel du DataFrame nettoyé avant export au format CSV.

In [25]:
df1['Identifiant_Véhicule'] = df1['Identifiant_Véhicule'].apply(lambda x: f'V{int(x):04d}')

In [26]:
# --- Aperçu des 15 premières observations du DataFrame nettoyé ---
df1.head(15)


,Identifiant_Véhicule,Année_Fabrication,Heures_Utilisation,Info_Itinéraire,Capacité_Charge,Charge_Réelle,Date_Dernier_Entretien,Type_Entretien,Température_Moteur,Pression_Pneus,...,Qualité_Huile,État_Freins,Historique_Pannes,Anomalies_Détectées,Score_Prédictif,Entretien_Nécessaire,Conditions_Météo,Conditions_Route,Etat_Batterie,Niveau_Risque
0,V0001,2020,3615,autoroute,5.969894,6.383313,2023-09-07,révision du moteur,112.5,117.1,...,84.369240,bon,0,1,0.5735,1,clair,urbain,63.6,Élevé
1,V0002,2021,128,urbain,41.784895,47.036831,2023-03-02,rotation des pneus,96.7,115.9,...,89.845122,mauvais,0,0,0.5804,1,clair,rural,84.1,Élevé
2,V0003,2016,6379,autoroute,2.909616,2.560338,2023-08-04,changement d'huile,107.1,117.1,...,91.277795,bon,1,1,0.6339,1,clair,rural,75.1,Élevé
3,V0004,2005,44,urbain,8.864533,6.675255,NaT,rotation des pneus,108.2,127.2,...,71.813017,mauvais,1,0,0.7480,1,clair,urbain,80.4,Élevé
4,V0005,2014,468,rural,21.902258,17.240408,2023-09-05,changement d'huile,121.4,123.6,...,78.009016,bon,1,1,0.7066,1,clair,autoroute,75.8,Élevé
5,V0006,2022,8650,autoroute,11.635688,12.462537,2023-04-03,changement d'huile,101.2,123.6,...,65.607610,bon,0,0,0.4174,0,clair,autoroute,89.3,Modéré
6,V0007,2020,521,urbain,20.693001,22.229658,2023-03-01,changement d'huile,104.5,122.1,...,57.331093,bon,1,0,0.5829,1,clair,rural,77.2,Élevé
7,V0008,2013,425,rural,18.440410,19.168108,2023-11-12,changement d'huile,114.9,109.7,...,86.712483,mauvais,1,1,0.8582,1,clair,rural,61.2,Critique
8,V0009,2021,3772,autoroute,41.770024,45.594913,NaT,changement d'huile,96.3,123.6,...,68.986462,mauvais,0,0,0.6233,1,pluvieux,autoroute,86.4,Élevé
9,V0010,2018,7223,rural,24.729445,17.352147,NaT,changement d'huile,104.1,124.1,...,77.728079,bon,0,0,0.4188,0,clair,urbain,75.4,Modéré


In [27]:
# --- Statistiques descriptives des variables catégorielles ---
df1.describe(include='object')


,Identifiant_Véhicule,Info_Itinéraire,Type_Entretien,État_Freins,Conditions_Météo,Conditions_Route,Niveau_Risque
count,3618,3618,3618,3618,3618,3618,3618
unique,3618,3,3,3,4,3,4
top,V0001,autoroute,changement d'huile,bon,clair,autoroute,Élevé
freq,1,1825,1660,1824,2926,1421,1570


In [28]:
# --- Statistiques descriptives des variables numériques (float) ---
df1.describe(include='float64')


,Capacité_Charge,Charge_Réelle,Température_Moteur,Pression_Pneus,Consommation_Carburant,État_Batterie,Niveaux_Vibration,Qualité_Huile,Score_Prédictif,Etat_Batterie
count,3618.000000,3618.000000,3618.000000,3618.000000,3618.000000,3618.000000,3618.000000,3618.000000,3618.000000,3618.000000
mean,18.308655,17.323311,105.286346,117.768546,46.516915,45.642286,7.035989,79.439685,0.591107,77.177391
std,14.238746,13.822624,9.125205,7.428233,5.663712,1.601952,2.591630,9.806641,0.169457,10.209970
min,0.017976,0.017867,79.100000,92.300000,32.000000,45.000000,1.240000,43.176358,0.045100,46.200000
25%,6.254339,5.765859,98.900000,113.600000,42.400000,45.000000,5.110000,72.927933,0.478475,69.900000
50%,15.093059,13.976995,105.100000,118.800000,46.800000,45.000000,6.925000,79.346645,0.603550,77.600000
75%,27.668727,26.122385,111.800000,123.600000,51.100000,45.000000,8.837500,86.193171,0.722150,84.800000
max,54.965729,65.029212,128.000000,132.000000,55.000000,50.000000,12.000000,100.000000,0.997300,100.000000


In [29]:
# --- Export du DataFrame nettoyé au format CSV ---
# Le fichier résultant servira de base au système RAG.

OUTPUT_CSV = r'C:\Users\ayaaf\OneDrive\Belgeler\truck_rag_sys\uploads\histoire_de_maintenance.csv'
df1.to_csv(OUTPUT_CSV, index=False)
print(f'Jeu de données exporté : {OUTPUT_CSV} ({len(df1)} lignes, {len(df1.columns)} colonnes)')


Jeu de données exporté : C:\Users\ayaaf\OneDrive\Belgeler\truck_rag_sys\uploads\histoire_de_maintenance.csv (3618 lignes, 23 colonnes)
